#Given a shapefile with polygons (not tiles)

Build tiles by intersecting with polygons.

You can choose tile size, how to intersect, etc.

#Import libraries

In [ ]:
!pip install rasterio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.3/22.3 MB 129.3 MB/s eta 0:00:00


In [ ]:
import cupy as cp
import rasterio, os
from rasterio.windows import Window
import numpy as np
from pathlib import Path
import geopandas as gpd
import rasterio
from shapely.geometry import box
import pandas as pd
from rasterio.features import rasterize
import warnings
from pyproj import Transformer
from shapely.validation import make_valid

#Merge if more than one shapefile

# Read both shapefiles
shapefile1 = gpd.read_file('/content/drive/MyDrive/salt_marsh/tiles/epa_project/agawam_tile_data/manual_tiling_labeled_medium_res/Labeled Irregular Polygons_ALL/LabeledData_ALL_2024.shp')
shapefile2 = gpd.read_file('/content/drive/MyDrive/salt_marsh/tiles/epa_project/agawam_tile_data/manual_tiling_labeled_medium_res/Labeled Square Polygones_ALL_1mx1m/LabeledData_All_2024_1mx1m_grid_2.shp')

# Check CRS (Coordinate Reference System)
print("CRS of first shapefile:", shapefile1.crs)
print("CRS of second shapefile:", shapefile2.crs)

# Make sure they have the same CRS
if shapefile1.crs != shapefile2.crs:
    # Reproject the second shapefile to match the first
    shapefile2 = shapefile2.to_crs(shapefile1.crs)

# Check column names to ensure compatibility
print("Columns in first shapefile:", shapefile1.columns.tolist())
print("Columns in second shapefile:", shapefile2.columns.tolist())

column_mapping_1 = {
    'class_typ': 'class_label',  # Example: Map 'class_typ' to 'class_label'
    'veg_id': 'vegetation_id'
}
shapefile1 = shapefile1.rename(columns=column_mapping_1)

column_mapping_2 = {
    'class_typ': 'class_label',  # Example: Map 'class_typ' to 'class_label'
    'veg_id': 'vegetation_id'
}
shapefile2 = shapefile2.rename(columns=column_mapping_2)

# Select just the columns you need from both
common_cols = ['geometry', 'class_label']  # Add your important columns
merged_shapefile = gpd.GeoDataFrame(pd.concat([
    shapefile1[common_cols],
    shapefile2[common_cols]
], ignore_index=True))

# Save the merged shapefile
merged_shapefile.to_file('/content/merged/merged_shapefile.shp')

In [ ]:
ortho_path =   '/content/drive/MyDrive/Cynosuroides and Terrapin Habitat_FWS/Agawam River/UAS Data/Orhtomosaics and DEMs/04Sep2025/04Sep2025_AGR_Mid_RGB_Ortho.tif'  #rgb
#shapefile_path = '/content/drive/MyDrive/Cynosuroides and Terrapin Habitat_FWS/Agawam River/In Situ Data/Ground Truth Data/Shapefiles/LabeledData_ALL_2024.shp' # .shp file
output_path =   '/content/drive/MyDrive/salt_marsh/turtles/tiles/manual_output/' #folder

#ortho_path =   '/content/drive/MyDrive/salt_marsh/tiles/epa_project/agawam_tile_data/manual_tiling_labeled_medium_res/03Sep2024_AGR_Low_Mavic3_Ortho.tif'  #rgb
#ortho_path =   '/content/drive/MyDrive/salt_marsh/tiles/epa_project/agawam_tile_data/manual_tiling_labeled_medium_res/03Sep2024_AGR_Low_Mavic3_DEM_HighestSettings.tif'   #dem
shapefile_path = '/content/drive/MyDrive/epa_project/agawam_tile_data/manual_tiling_labeled_medium_res/Labeled Irregular Polygons_ALL/LabeledData_ALL_2024.shp' # .shp file
#output_path =   '/content/drive/MyDrive/salt_marsh/tiles/epa_project/agawam_tile_data/manual_tiling_labeled_medium_res/manual_output/' #folder

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
epa_base = '/content/drive/MyDrive/salt_marsh/turtles/tiles/'

Mounted at /content/drive


#RFM (shouldn't need this)

In [ ]:
epa_base = '/content/drive/MyDrive/salt_marsh/turtles/tiles/'  #I believe this is where you have to use my shortcut

In [ ]:
sites = [ '02Aug19_OTH_Mid_Mica_Ortho',  #10M
          '04Aug21_OTH_Low_Mica',
          '01Aug20_OTH_MidOut_Mica_Ortho',  #10M
          '24Aug21_OTH_Mid_Mica_Ortho',
          #'10Aug22_RR_Low_Mavic_Ortho',  #only 3 bands
          #'02May2022_OTH_Low_Mavic_ortho',
          '08Aug19_RR_Low_Mica_Ortho',
          '06Aug19_RR_High_Mica_Ortho',
          '07Jul21_RR_Low_Mica_Ortho',
]
site = sites[1]
site

'04Aug21_OTH_Low_Mica'

In [ ]:
the_ortho = f'Orthomosaics/{site}.tif'
the_ortho = f'{site}/{site}.tif'

In [ ]:
shape_file = 'OTH_shape/OTH_Poygons_RCW_20Jul2023_4326.shp'
#shape_file = 'RR_shape/RR_Polygons_Final_v3_reclassed.shp'

In [ ]:
ortho_path = epa_base + 'manual_output/' + the_ortho
shapefile_path = epa_base + shape_file
ortho_path, shapefile_path

('/content/drive/MyDrive/salt_marsh/tiles/epa_project/RFM_project/manual_output/04Aug21_OTH_Low_Mica/04Aug21_OTH_Low_Mica.tif',
 '/content/drive/MyDrive/salt_marsh/tiles/epa_project/RFM_project/OTH_shape/OTH_Poygons_RCW_20Jul2023_4326.shp')

In [ ]:
output_path = f'/content/drive/MyDrive/salt_marsh/tiles/epa_project/RFM_project/manual_output/{site}/'

In [ ]:
!nvidia-smi

Mon Oct 20 18:14:13 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   36C    P0             53W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
!pip install cupy-cuda12x

In [ ]:
!pip install rasterio geopandas pandas numpy shapely

In [ ]:
epa_base+'manual_output/'

'/content/drive/MyDrive/salt_marsh/turtles/tiles/manual_output/'

In [ ]:
os.listdir(epa_base+'manual_output/')

['all_tiles_rgb_62.npy',
 'unlabeled_tile_metadata_rgb_62.csv',
 'tile_grid_rgb_62_centroid_7.shp',
 'tile_grid_rgb_62_centroid_7.shx',
 'tile_grid_rgb_62_centroid_7.dbf',
 'tile_grid_rgb_62_centroid_7.cpg',
 'tile_grid_rgb_62_centroid_7.prj',
 'training_labels_rgb_62_centroid_7.csv']

In [ ]:
os.listdir(epa_base+'manual_output/'+site)

['04Aug21_OTH_Low_Mica.tif']

In [ ]:
# Load shapefile and see all column names
if shapefile_path:
    shapes_gdf = gpd.read_file(shapefile_path)
    print("Available columns in shapefile:")
    print(shapes_gdf.columns)

    # You can also see sample data:
    print("\nFirst row of data:")
    print(shapes_gdf.iloc[0])

Available columns in shapefile:
Index(['id', 'Class', 'Area (m^2)', 'geometry'], dtype='object')

First row of data:
id                                                           21
Class                                                         1
Area (m^2)                                               19.333
geometry      POLYGON Z ((359268.964 4624961.239 0.455, 3592...
Name: 0, dtype: object


###Make sure same CRS

In [ ]:
with rasterio.open(ortho_path) as src:
    ortho_crs = src.crs
    print(f"Orthomosaic CRS: {ortho_crs}")
    print(f"Number of bands: {src.count}")
    print(f"Data type: {src.dtypes}")

    # Get band descriptions and names
    print("\nBand information:")
    for i in range(1, src.count + 1):
        band_description = src.descriptions[i-1] or f"Band {i}"

        # Check for colorinterp (sometimes indicates band type like R,G,B)
        color_interp = src.colorinterp[i-1].name

        # Get band statistics
        band_data = src.read(i)
        min_val = band_data.min()
        max_val = band_data.max()
        mean_val = band_data.mean()

        print(f"  Band {i}:")
        print(f"    Description: {band_description}")
        print(f"    Color interpretation: {color_interp}")
        print(f"    Min: {min_val}, Max: {max_val}, Mean: {mean_val:.2f}")

        # Get band tags (metadata)
        tags = src.tags(i)
        if tags:
            print(f"    Metadata tags: {tags}")

Orthomosaic CRS: EPSG:26919
Number of bands: 3
Data type: ('uint8', 'uint8', 'uint8')

Band information:
  Band 1:
    Description: Band 1
    Color interpretation: red
    Min: 0, Max: 255, Mean: 209.13
  Band 2:
    Description: Band 2
    Color interpretation: green
    Min: 0, Max: 255, Mean: 211.31
  Band 3:
    Description: Band 3
    Color interpretation: blue
    Min: 0, Max: 255, Mean: 201.47


In [ ]:
# Inside your create_qgis_grid method, after reading the shapefile:
shapes_gdf = gpd.read_file(shapefile_path)
print(f"Original shapefile CRS: {shapes_gdf.crs}")


Original shapefile CRS: EPSG:26919


In [ ]:
try:
    # Try to get GPU device info
    device_info = cp.cuda.runtime.getDeviceProperties(0)
    print(f"GPU available: {device_info['name']}")
except:
    print("No GPU available - using NumPy instead")
    cp = np  #default to numpy


GPU available: b'NVIDIA A100-SXM4-80GB'


In [ ]:
def calculate_tile_size(ortho_path, target_meters=1.0):
    """
    Calculate pixel dimensions needed for a tile of target_meters x target_meters,
    automatically handling different CRS types.

    Args:
        ortho_path: Path to orthomosaic
        target_meters: Desired tile size in meters (default 1.0 for 1m x 1m)

    Returns:
        tuple: (tile_width, tile_height) in pixels
    """
    with rasterio.open(ortho_path) as src:
        transform = src.transform

        if src.crs.is_geographic:  # If in degrees (like WGS84)
            # Get center of image for reasonable conversion
            center_x = src.bounds.left + (src.bounds.right - src.bounds.left)/2
            center_y = src.bounds.bottom + (src.bounds.top - src.bounds.bottom)/2

            # Convert to UTM for meter calculations
            utm_zone = int((center_x + 180) / 6) + 1
            transformer = Transformer.from_crs(
                src.crs,
                f"+proj=utm +zone={utm_zone} +datum=WGS84",
                always_xy=True
            )

            # Calculate pixel size in meters
            x1, y1 = transformer.transform(center_x, center_y)
            x2, y2 = transformer.transform(center_x + transform.a, center_y + transform.e)

            x_res = abs(x2 - x1)  # meters per pixel
            y_res = abs(y2 - y1)  # meters per pixel

        else:  # If already in projected CRS (meters or feet)
            # Convert to meters if needed
            units = src.crs.linear_units
            if units == 'meter' or units == 'metre':
                x_res = abs(transform.a)
                y_res = abs(transform.e)
            elif units == 'foot':
                x_res = abs(transform.a) * 0.3048  # convert feet to meters
                y_res = abs(transform.e) * 0.3048
            else:
                raise ValueError(f"Unsupported linear units: {units}")

        # Calculate pixels needed for target size
        tile_width = round(target_meters / x_res)
        tile_height = round(target_meters / y_res)

        return max(tile_width, tile_height), max(tile_width, tile_height)  #make sure square


tile_width, tile_height = calculate_tile_size(ortho_path, target_meters=2.0)
print(f" need {tile_width} x {tile_height} pixels")

 need 82 x 82 pixels


In [ ]:
threshold = .7  #overlap necessary to inherit shape class

In [ ]:
band = 'rgb'  #mica5 BGR RE NIR
method = 'centroid'

###Notes

There are three methods for intersecting tiles with shapes:

1. pixel-based. How many pixels intersect, percentage wise, between tile and shape?
2. Area-based. Geometric intersection.
3. Centroid-based. Does centroid of image intersect with shape?

If tiles are found that intersect, they inherit the label of the shape.

In all 3 cases, there is a check for bogus tile data for tiles that intersect. If bogus data is found, the class given is the shape class + 100, e.g., 17 becomes 117.

In [ ]:
legal_classes = [1,3,4,5,6,12,17,18,19,20,99] #[3,4,5]
legal_class_names = '1,3,4,5,6,12,17,18,19,20,99'

In [ ]:
class_column = 'Class' #'ReClass v4'

In [ ]:
class GPUTileProcessor:
    def __init__(self, ortho_path: str, tile_height: int, tile_width: int, output_dir: str, band_identifier="multiband"):
        """
        Initialize GPU-accelerated tile processor.

        Args:
            ortho_path: Path to orthomosaic
            tile_height: Height of tiles in pixels
            tile_width: Width of tiles in pixels
            output_dir: Directory to save data
            band_identifier: Identifier for the band configuration (e.g., "5band", "RGB+NIR+RE")
        """
        self.ortho_path = ortho_path
        self.tile_height = tile_height
        self.tile_width = tile_width
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.bogus_tiles = []  #tiles who intersect but have unusable data - 100 added to class
        self.band_identifier = band_identifier

    def process_tiles(self):
        """Extract tiles from orthomosaic using GPU acceleration."""
        with rasterio.open(self.ortho_path) as src:
            width = src.width
            height = src.height
            bands = src.count
            print(f'{width=}, {height=}, {bands=}')

            n_tiles_x = int(np.ceil(width / self.tile_width))
            n_tiles_y = int(np.ceil(height / self.tile_height))

            # Initialize array on GPU
            all_tiles = cp.zeros((n_tiles_x * n_tiles_y, bands,
                                self.tile_height, self.tile_width),
                                dtype=src.dtypes[0])

            # Read the entire image into GPU memory
            full_image = cp.array(src.read())

            tile_metadata = []
            tile_idx = 0

            # Process tiles in batches
            batch_size = 1000  # Adjust based on your GPU memory

            for yi in range(n_tiles_y):
                if yi % 100 == 0:
                    print(f"Processing row {yi}/{n_tiles_y}")
                for xi in range(0, n_tiles_x, batch_size):
                    batch_end = min(xi + batch_size, n_tiles_x)

                    for batch_xi in range(xi, batch_end):
                        x_start = batch_xi * self.tile_width
                        y_start = yi * self.tile_height

                        x_end = min(x_start + self.tile_width, width)
                        y_end = min(y_start + self.tile_height, height)

                        actual_height = y_end - y_start
                        actual_width = x_end - x_start

                        # Extract tile on GPU
                        tile_data = full_image[:, y_start:y_end, x_start:x_end]

                        # Store in the big array
                        all_tiles[tile_idx, :, :actual_height, :actual_width] = tile_data

                        # Get geographic coordinates
                        window = Window(x_start, y_start,
                                      actual_width, actual_height)
                        geo_bounds = rasterio.windows.bounds(window, src.transform)

                        # Store metadata
                        tile_metadata.append({
                            'tile_id': f"{batch_xi}_{yi}",
                            'x_coord': batch_xi,
                            'y_coord': yi,
                            'tile_idx': tile_idx,
                            'bbox': geo_bounds,
                            'actual_height': actual_height,
                            'actual_width': actual_width
                        })

                        tile_idx += 1

            # Move data back to CPU for saving
            array_path = self.output_dir / f'all_tiles_{self.band_identifier}_{self.tile_width}.npy'
            np_convert = cp.asnumpy(all_tiles)
            np.save(array_path, np_convert)
            print("Image min/max", np_convert.min(), np_convert.max())

            metadata_df = pd.DataFrame(tile_metadata)
            metadata_df.to_csv(self.output_dir / f'unlabeled_tile_metadata_{self.band_identifier}_{self.tile_width}.csv', index=False)

            return metadata_df, array_path

    def process_tiles_memmap(self):
        """Process tiles using memory mapping for very large datasets."""
        with rasterio.open(self.ortho_path) as src:
            width = src.width
            height = src.height
            bands = src.count

            n_tiles_x = int(np.ceil(width / self.tile_width))
            n_tiles_y = int(np.ceil(height / self.tile_height))

            # Create memory-mapped array
            array_path = self.output_dir / f'all_tiles_{self.band_identifier}_{self.tile_width}.npy'
            all_tiles = np.lib.format.open_memmap(
                array_path,
                mode='w+',
                dtype=src.dtypes[0],
                shape=(n_tiles_x * n_tiles_y, bands, self.tile_height, self.tile_width)
            )

            chunk_size = 1000
            tile_metadata = []
            tile_idx = 0

            for yi in range(n_tiles_y):
                for xi in range(0, n_tiles_x, chunk_size):
                    chunk_end = min(xi + chunk_size, n_tiles_x)
                    chunk_width = (chunk_end - xi) * self.tile_width

                    # Read chunk
                    window = Window(xi * self.tile_width, yi * self.tile_height,
                                  chunk_width, self.tile_height)
                    chunk_data = cp.array(src.read(window=window))

                    for batch_xi in range(xi, chunk_end):
                        rel_x = (batch_xi - xi) * self.tile_width
                        x_start = batch_xi * self.tile_width
                        y_start = yi * self.tile_height

                        x_end = min(rel_x + self.tile_width, chunk_width)
                        actual_height = min(self.tile_height, height - y_start)
                        actual_width = min(self.tile_width, width - x_start)

                        # Extract tile
                        tile_data = chunk_data[:, :actual_height, rel_x:rel_x+actual_width]

                        # Store tile
                        all_tiles[tile_idx, :, :actual_height, :actual_width] = cp.asnumpy(tile_data)

                        # Store metadata
                        window = Window(x_start, y_start, actual_width, actual_height)
                        geo_bounds = rasterio.windows.bounds(window, src.transform)

                        tile_metadata.append({
                            'tile_id': f"{batch_xi}_{yi}",
                            'x_coord': batch_xi,
                            'y_coord': yi,
                            'tile_idx': tile_idx,
                            'bbox': geo_bounds,
                            'actual_height': actual_height,
                            'actual_width': actual_width
                        })

                        tile_idx += 1

            metadata_df = pd.DataFrame(tile_metadata)
            metadata_df.to_csv(self.output_dir / f'tile_metadata_{self.band_identifier}_{self.tile_width}.csv', index=False)

            return metadata_df, array_path

    def create_qgis_grid(self, metadata_df: pd.DataFrame, shapefile_path=None,
                     class_column=None, method='centroid', threshold=0.5):
        """
        Create a  QGIS-compatible shapefile of the tile grid.

        Args:
            metadata_df: DataFrame with tile metadata
            shapefile_path: shapefile_path: Path to shapefile with classifications. If None, creates
                    an unclassified grid with empty 'label' field.
            class_column: Column name in shapefile containing class labels
            method: 'centroid', 'area', or 'pixel'
            threshold: Threshold for area or pixel methods (default 0.5)
        """
        if shapefile_path is None and class_column is not None:
            warnings.warn("class_column provided but no shapefile_path - no classification will be performed")

        # Create geometries for each tile
        print("Creating geometries ...")
        geometries = []
        for i, row in metadata_df.iterrows():
            if int(i) % 100000 == 0:
                print(f"Processing tile {i}/{len(metadata_df)}")
            bbox = row['bbox']
            # If bbox is stored as a string or list with many elements
            if isinstance(bbox, str):
                # Parse string representation - adjust parsing based on actual format
                coords = eval(bbox)  # Only use if bbox is a string representation of a tuple
                geometries.append(box(*coords))
            elif hasattr(bbox, '__iter__') and len(bbox) == 4:
                # If it's already an iterable with four elements
                geometries.append(box(*bbox))
            else:
                raise TypeError(f"Unexpected bbox format: {bbox} of type {type(bbox)}")

        # Rename columns to be shapefile-compatible (max 10 chars)
        metadata_df = metadata_df.copy()
        column_mapping = {
            'actual_height': 'act_height',
            'actual_width': 'act_width',
            'tile_idx': 'tile_idx',
            'x_coord': 'x_coord',
            'y_coord': 'y_coord',
            'tile_id': 'tile_id',
            'bbox': 'bbox'
        }
        metadata_df = metadata_df.rename(columns=column_mapping)

        # Create GeoDataFrame with original CRS
        print("Creating GeoDataFrame ...")
        with rasterio.open(self.ortho_path) as src:
            original_crs = src.crs
            gdf = gpd.GeoDataFrame(
                metadata_df,
                geometry=geometries,
                crs=original_crs
            )

        # Add label field (initially None)
        gdf['label'] = None

        if shapefile_path:
            # Read shapefile
            shapes_gdf = gpd.read_file(shapefile_path)
            if 'Raw Subcla' in shapes_gdf.columns:
                shapes_gdf[class_column] = shapes_gdf['Raw Subcla'].apply(transform_value)
                #shapes_gdf[f'ReClass v4'] = shapes_gdf['Raw Subcla'].apply(transform_value)
            # Check for invalid geometries
            invalid_count = sum(~shapes_gdf.geometry.is_valid)
            print(f"Found {invalid_count} invalid geometries")

            if invalid_count > 0:
                # Fix invalid geometries
                shapes_gdf['geometry'] = shapes_gdf.geometry.apply(make_valid)

                # Verify fix
                still_invalid = sum(~shapes_gdf.geometry.is_valid)
                print(f"After repair: {still_invalid} invalid geometries remain")

            if class_column not in shapes_gdf.columns:
                raise ValueError(f"Column '{class_column}' not found in shapefile. Available columns: {shapes_gdf.columns.tolist()}")

            if method == 'centroid':
                gdf = self._apply_centroid_method(gdf, shapes_gdf, class_column)
            elif method == 'area':
                gdf = self._apply_area_method(gdf, shapes_gdf, class_column, threshold)
            elif method == 'pixel':
                gdf = self._apply_pixel_method(gdf, shapes_gdf, class_column, threshold)
            else:
                raise ValueError(f"Unknown method: {method}. Use 'centroid', 'area', or 'pixel'")
        else:
            print("No shapefile provided. Creating unclassified grid.")
            method = 'unclassified'

        # Save grid shapefile in original CRS
        grid_path = self.output_dir / f'tile_grid_{self.band_identifier}_{self.tile_width}_{method}_{str(threshold)[2:]}.shp'
        gdf.to_file(grid_path)

        return grid_path

    def _check_if_bogus_tile(self, tile_data, tile_idx, tile_id, original_label):
        """
        Check if a tile contains usable data or is bogus.
        Returns: (is_bogus, reason)
        """
        num_bands = tile_data.shape[0]

        # DEM or single-band case
        if num_bands == 1:
            if np.all(tile_data == -32767.0) or np.allclose(tile_data, tile_data.mean(), atol=1.0):
                return True, "-32767.0 DEM"
            if np.allclose(tile_data, tile_data.mean(), atol=1.0):
                return True, "all same DEM"
            return False, None

        # RGB case
        elif num_bands == 3:
            # Check if effectively grayscale
            is_grayscale = np.allclose(tile_data[0], tile_data[1], atol=1.0) and \
                           np.allclose(tile_data[1], tile_data[2], atol=1.0)
            if is_grayscale:
                return True, "all RGB bands close/identical"

            # Check for appropriate range across all bands
            td_max = np.max(tile_data)
            # Assuming 16-bit multispectral data (adjust if different)
            if td_max > 65535:
                return True, f"max band value {td_max} > 65535"

            # Check color saturation
            rgb_bands = tile_data[:3]
            if td_max > 1:
                # Normalize based on data range
                if td_max > 255 and td_max <= 65535:
                    normalized = rgb_bands / 65535.0  # 16-bit
                else:
                    normalized = rgb_bands / 255.0    # 8-bit
            else:
                normalized = rgb_bands  # Already normalized

            rgb_min = np.min(normalized, axis=0)
            rgb_max = np.max(normalized, axis=0)
            saturation = np.mean((rgb_max - rgb_min) / (rgb_max + 0.0001))

            if saturation < 0.05:
                return True, "low RGB saturation"

        # 5-band case (RGB+NIR+RedEdge)
        elif num_bands == 5:
            # Check RGB bands for grayscale
            is_grayscale = np.allclose(tile_data[0], tile_data[1], atol=1.0) and \
                           np.allclose(tile_data[1], tile_data[2], atol=1.0)
            if is_grayscale:
                return True, "all RGB bands close/identical"

            # Check for appropriate range across all bands
            td_max = np.max(tile_data)
            # Assuming 16-bit multispectral data (adjust if different)
            if td_max > 65535:
                return True, f"max band value {td_max} > 65535"

            # Check RGB bands for color saturation
            rgb_bands = tile_data[:3]
            if td_max > 1:
                # Normalize based on data range
                if td_max > 255 and td_max <= 65535:
                    normalized = rgb_bands / 65535.0  # 16-bit
                else:
                    normalized = rgb_bands / 255.0    # 8-bit
            else:
                normalized = rgb_bands  # Already normalized

            rgb_min = np.min(normalized, axis=0)
            rgb_max = np.max(normalized, axis=0)
            saturation = np.mean((rgb_max - rgb_min) / (rgb_max + 0.0001))

            if saturation < 0.05:
                return True, "low RGB saturation"

            # Check NIR and Red-Edge bands
            nir_band = tile_data[3]
            red_edge_band = tile_data[4]

            # Check for empty or invalid data in these bands
            if np.all(nir_band == 0) or np.all(red_edge_band == 0):
                return True, "empty NIR or Red-Edge band"

        # Handle any other number of bands with basic checks
        else:
            # Basic check for empty or constant data
            if np.all(tile_data == 0) or np.all(tile_data == tile_data[0, 0, 0]):
                return True, f"empty or constant {num_bands}-band data"

        return False, None

    def _apply_centroid_method(self, gdf, shapes_gdf, class_column):
        """Apply centroid-based classification with proper projection."""
        # Load the tile array
        print('Starting centroid intersection ...')
        tiles = np.load(self.output_dir / f'all_tiles_{self.band_identifier}_{self.tile_width}.npy')

        # Get the current CRS
        current_crs = gdf.crs

        # Check if we're in a geographic CRS
        if current_crs and current_crs.is_geographic:
            # Get bounds to determine appropriate UTM zone
            bounds = gdf.total_bounds
            center_lon = (bounds[0] + bounds[2]) / 2
            center_lat = (bounds[1] + bounds[3]) / 2

            # Calculate UTM zone from the center longitude
            utm_zone = int((center_lon + 180) / 6) + 1
            hemisphere = 'north' if center_lat >= 0 else 'south'
            epsg = 32600 + utm_zone if hemisphere == 'north' else 32700 + utm_zone
            utm_crs = f"EPSG:{epsg}"

            # Project both geodataframes to UTM for accurate centroid calculation
            gdf_utm = gdf.to_crs(utm_crs)
            shapes_utm = shapes_gdf.to_crs(utm_crs)

            # Calculate centroids in UTM
            tile_centroids = gdf_utm.geometry.centroid

            # Project centroids back to original CRS for intersection
            # We create a GeoDataFrame with the centroids and project it back
            centroids_gdf = gpd.GeoDataFrame(geometry=tile_centroids, crs=utm_crs)
            centroids_gdf = centroids_gdf.to_crs(current_crs)
            tile_centroids = centroids_gdf.geometry
        else:
            # If already in a projected CRS, calculate centroids directly
            tile_centroids = gdf.geometry.centroid

        # Initialize bogus_tiles list if it doesn't exist
        if not hasattr(self, 'bogus_tiles'):
            self.bogus_tiles = []

        # Do intersection
        for idx, centroid in enumerate(tile_centroids):
            if idx % 100000 == 0:
                  print(f"Processing tile {idx}/{len(tile_centroids)}")
            containing_shape = shapes_gdf[shapes_gdf.contains(centroid)]
            if len(containing_shape) > 0:
                # Get the original label
                original_label = containing_shape.iloc[0][class_column]

                # Get tile data to check if it's bogus
                tile_idx = gdf.iloc[idx]['tile_idx']
                tile_data = tiles[tile_idx]
                tile_id = gdf.iloc[idx]['tile_id']

                # Check if the tile is bogus
                is_bogus, reason = self._check_if_bogus_tile(tile_data, tile_idx, tile_id, original_label)

                #if is_bogus:
                #    gdf.at[idx, 'label'] = original_label + 1000
                #    self.bogus_tiles.append((tile_idx, tile_id, original_label, reason))
                #else:
                gdf.at[idx, 'label'] = original_label

        return gdf

    def _apply_area_method(self, gdf, shapes_gdf, class_column, threshold):
        """
        Apply area-based classification without reprojection.
        Adds special class codes (+100) for tiles that have no data.
        """
        # Load the tile array for checking pixel values
        tiles = np.load(self.output_dir / f'all_tiles_{self.band_identifier}_{self.tile_width}.npy')

        for idx, tile in gdf.iterrows():
            if idx % 1000 == 0:
                print(f"Processing tile {idx}/{len(gdf)}")
            tile_geom = tile.geometry
            tile_area = tile_geom.area

            # Find all intersecting shapes
            intersecting_shapes = shapes_gdf[shapes_gdf.intersects(tile_geom)]

            if len(intersecting_shapes) > 0:
                # Calculate intersection areas
                areas = []
                for _, shape in intersecting_shapes.iterrows():
                    try:
                        intersection = tile_geom.intersection(shape.geometry)
                        fraction = intersection.area / tile_area
                        if fraction >= threshold:
                            areas.append((shape[class_column], fraction))
                    except Exception as e:
                        print(f"Skipping shape due to error: {e}")
                        continue

                # If we found any significant intersections
                if areas:
                    # Take the class with largest intersection
                    areas.sort(key=lambda x: x[1], reverse=True)
                    original_label = areas[0][0]

                    tile_idx = tile['tile_idx']
                    tile_data = tiles[tile_idx]
                    tile_id = tile['tile_id']

                    # Check if the tile is bogus
                    is_bogus, reason = self._check_if_bogus_tile(tile_data, tile_idx, tile_id, original_label)

                    if is_bogus:
                        gdf.at[idx, 'label'] = original_label + 1000
                        self.bogus_tiles.append((tile_idx, tile_id, original_label, reason))
                    else:
                        gdf.at[idx, 'label'] = original_label

        return gdf

    def _apply_pixel_method(self, gdf, shapes_gdf, class_column, threshold):
        """Apply pixel-based classification with checks for bogus images."""
        # Load the tile array
        tiles = np.load(self.output_dir / f'all_tiles_{self.band_identifier}_{self.tile_width}.npy')

        # Initialize bogus_tiles list if it doesn't exist
        if not hasattr(self, 'bogus_tiles'):
            self.bogus_tiles = []

        with rasterio.open(self.ortho_path) as src:
            # Create value mapping
            shapes = []
            value_to_label = {}
            for i, (_, shape) in enumerate(shapes_gdf.iterrows()):
                shapes.append((shape.geometry, i+1))
                value_to_label[i+1] = shape[class_column]

            # Rasterize all shapes
            full_mask = rasterize(
                shapes,
                out_shape=(src.height, src.width),
                transform=src.transform,
                dtype='int32'
            )

            # For each tile
            for idx, tile in gdf.iterrows():
                x_start = int(tile['x_coord'] * self.tile_width)
                y_start = int(tile['y_coord'] * self.tile_height)

                # Extract tile portion of the mask
                tile_mask = full_mask[
                    y_start:y_start + self.tile_height,
                    x_start:x_start + self.tile_width
                ]

                # Count pixels of each class
                unique, counts = np.unique(tile_mask, return_counts=True)
                pixel_counts = dict(zip(unique, counts))

                # Remove background (0)
                if 0 in pixel_counts:
                    del pixel_counts[0]

                if pixel_counts:
                    max_value = max(pixel_counts.items(), key=lambda x: x[1])
                    if max_value[1] / tile_mask.size >= threshold:
                        # Get the original label
                        original_label = value_to_label[max_value[0]]

                        # Get tile data to check if it's bogus
                        tile_idx = tile['tile_idx']
                        tile_data = tiles[tile_idx]
                        tile_id = tile['tile_id']

                        # Check if the tile is bogus
                        is_bogus, reason = self._check_if_bogus_tile(tile_data, tile_idx, tile_id, original_label)

                        if is_bogus:
                            gdf.at[idx, 'label'] = original_label + 1000
                            self.bogus_tiles.append((tile_idx, tile_id, original_label, reason))
                        else:
                            gdf.at[idx, 'label'] = original_label

        return gdf

    def export_training_csv(self, grid_shapefile, method="", threshold=0.5):
        """
        Export CSV mapping tile indices to labels from QGIS-edited shapefile.
        Includes all tiles, with NaN for unlabeled tiles.
        """
        # Read the edited shapefile
        gdf = gpd.read_file(grid_shapefile)

        # Create training data mapping for all tiles
        training_data = gdf[['tile_id', 'x_coord', 'y_coord', 'tile_idx', 'label']]

        # Save to CSV (NaN values handled automatically)
        csv_path = self.output_dir / f'training_labels_{self.band_identifier}_{self.tile_width}_{method}_{str(threshold)[2:]}.csv'
        training_data.to_csv(csv_path, index=False)

        # Print summary
        labeled_count = training_data['label'].notna().sum()
        total_count = len(training_data)
        print(f"Exported {total_count} tiles total:")
        print(f"  - {labeled_count} labeled")
        print(f"  - {total_count - labeled_count} unlabeled (NaN)")

        return csv_path

In [ ]:
import gc
gc.collect()

0

In [ ]:
# Initialize processor
processor = GPUTileProcessor(
    ortho_path=ortho_path,
    tile_height=max(tile_width, tile_height),
    tile_width=max(tile_width, tile_height),
    output_dir=output_path, #numpy array stored here,
    band_identifier=band
)

###Either build np tile grid or load if already exists

In [ ]:
import os
file_path =  f'{output_path}all_tiles_{band}_{tile_width}.npy'
if not os.path.exists(file_path):
  # Process tiles and create metadata
  print("Processing tiles ...")
  metadata_df, array_path = processor.process_tiles()
else:
  print("Loading tiles from file ...")
  metadata_df = pd.read_csv(f'{output_path}unlabeled_tile_metadata_{band}_{tile_width}.csv')
  array_path = file_path

Processing tiles ...
width=70154, height=51723, bands=3
Processing row 0/631
Processing row 100/631
Processing row 200/631
Processing row 300/631
Processing row 400/631
Processing row 500/631
Processing row 600/631
Image min/max 0 255


In [ ]:
metadata_df.head()

,tile_id,x_coord,y_coord,tile_idx,bbox,actual_height,actual_width
0,0_0,0,0,0,"(358378.28527672135, 4625384.414092664, 358380...",82,82
1,1_0,1,0,1,"(358380.28080412134, 4625384.414092664, 358382...",82,82
2,2_0,2,0,2,"(358382.27633152134, 4625384.414092664, 358384...",82,82
3,3_0,3,0,3,"(358384.27185892133, 4625384.414092664, 358386...",82,82
4,4_0,4,0,4,"(358386.2673863213, 4625384.414092664, 358388....",82,82


In [ ]:
len(metadata_df)

540136

In [ ]:
array_path

PosixPath('/content/drive/MyDrive/salt_marsh/turtles/tiles/manual_output/all_tiles_rgb_82.npy')

In [ ]:
# Clear memory cache
cp.get_default_memory_pool().free_all_blocks()
cp.get_default_pinned_memory_pool().free_all_blocks()

# Synchronize CUDA stream
cp.cuda.stream.get_current_stream().synchronize()
gc.collect()

0

###7 minutes

In [ ]:

# Usage:
"""
# Use with centroid method (original behavior)
grid_path = processor.create_qgis_grid(
    metadata_df=metadata_df,
    shapefile_path='classifications.shp',
    class_column='class_label',
    method='centroid'
)

# Or use area-based method
grid_path = processor.create_qgis_grid(
    metadata_df=metadata_df,
    shapefile_path='classifications.shp',
    class_column='class_label',
    method='area',
    threshold=0.5
)

# Or use pixel-based method
grid_path = processor.create_qgis_grid(
    metadata_df=metadata_df,
    shapefile_path='classifications.shp',
    class_column='class_label',
    method='pixel',
    threshold=0.5
)
"""

In [ ]:
# Create QGIS grid with pre-populated labels from shapefile
#grid_path = processor.create_qgis_grid(metadata_df, shapefile_path, 'ReClass v4')
grid_path = processor.create_qgis_grid(metadata_df, shapefile_path, class_column, method=method, threshold=threshold)
#grid_path = processor.create_qgis_grid(metadata_df, shapefile_path, 'ReClass v4', method='pixel', threshold=.5)
#grid_path = processor.create_qgis_grid(metadata_df)


Creating geometries ...
Processing tile 0/540136
Processing tile 100000/540136
Processing tile 200000/540136
Processing tile 300000/540136
Processing tile 400000/540136
Processing tile 500000/540136
Creating GeoDataFrame ...
Found 3 invalid geometries
After repair: 0 invalid geometries remain
Starting centroid intersection ...
Processing tile 0/540136
Processing tile 100000/540136
Processing tile 200000/540136
Processing tile 300000/540136
Processing tile 400000/540136
Processing tile 500000/540136


In [ ]:
bogus_tiles = processor.bogus_tiles
len(bogus_tiles)

2325

In [ ]:
bogus_tiles  #given label of original+100

In [ ]:
grid_path

PosixPath('/content/drive/MyDrive/salt_marsh/turtles/tiles/manual_output/tile_grid_rgb_82_centroid_7.shp')

In [ ]:
# After editing in QGIS, export training data
training_csv = processor.export_training_csv(grid_path, method=method, threshold=threshold)


Exported 540136 tiles total:
  - 17977 labeled
  - 522159 unlabeled (NaN)


In [ ]:
labels = pd.read_csv(f'{output_path}training_labels_{band}_{tile_width}_{method}_{str(threshold)[2:]}.csv')
labels['label'].value_counts()

,count
label,
22.0,7214
91.0,4101
96.0,1554
92.0,1108
12.0,1093
6.0,1051
97.0,490
95.0,489
17.0,321


In [ ]:
foobar()

In [ ]:

# Load data for training:
tiles = np.load(f'{output_path}all_tiles_{band}_{tile_width}.npy')


# Get specific tile using its index:
tile_idx = labels.loc[0, 'tile_idx']
tile_data = tiles[tile_idx]
tile_data.shape

In [ ]:
labels.head()

In [ ]:
os.listdir('/content/drive/MyDrive/salt_marsh/tiles/epa_project/RFM_project/manual_output/24Aug21_OTH_Mid_Mica_Ortho/')

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Load the labels and tile data
labels_df = pd.read_csv(f'{output_path}training_labels_{band}_{tile_width}_{method}_{str(threshold)[2:]}.csv')
print(f'{labels_df.columns=}')
tiles = np.load(f'{output_path}all_tiles_{band}_{tile_width}.npy')

# Find any labeled tile (not NaN)
labeled_tiles = labels_df[labels_df['label'].notna()]
if len(labeled_tiles) > 0:
    # Get first labeled tile
    example_tile = labeled_tiles.iloc[0]
    tile_idx = example_tile['tile_idx']
    tile_label = example_tile['label']

    # Get the tile data
    tile_data = tiles[tile_idx]

    # Determine if DEM or RGB based on number of bands
    num_bands = tile_data.shape[0]
    is_rgb = num_bands >= 3

    print(f"Number of bands: {num_bands}")
# Determine data type based on number of bands
num_bands = tile_data.shape[0]
print(f"Number of bands: {num_bands}")

if num_bands >= 3:
    # For 5-band data, we need to select which bands to display
    plt.figure(figsize=(15, 10))

    # Option 1: Show true color (RGB) composite
    plt.subplot(1, 2, 1)

    # Create an empty array for the RGB image
    rgb_image = np.zeros((tile_data.shape[1], tile_data.shape[2], 3))

    # Account for RBG order: map to RGB for display
    rgb_image[:,:,0] = tile_data[0]  # R → R
    rgb_image[:,:,1] = tile_data[2]  # G → G
    rgb_image[:,:,2] = tile_data[1]  # B → B

    # Normalize the data for display if needed
    if rgb_image.max() > 1.0:
        rgb_image = rgb_image / rgb_image.max()

    plt.imshow(rgb_image)
    plt.title("RGB True Color Composite")
    plt.axis('off')

    # Option 2: Show false color composite with NIR
    plt.subplot(1, 2, 2)

    # For the full 5-band RBG-RE-NIR, create a false color composite
    # Assuming band order: Red(0), Blue(1), Green(2), RedEdge(3), NIR(4)
    false_color = np.zeros((tile_data.shape[1], tile_data.shape[2], 3))
    false_color[:,:,0] = tile_data[4]  # NIR in red channel
    false_color[:,:,1] = tile_data[0]  # Red in green channel
    false_color[:,:,2] = tile_data[2]  # Green in blue channel

    # Normalize the data for display
    if false_color.max() > 1.0:
        false_color = false_color / false_color.max()

    plt.imshow(false_color)
    plt.title("False Color (NIR-R-G)")
    plt.axis('off')

else:
    # DEM data or single band - use first band with terrain colormap
    tile_image = tile_data[0]
    plt.figure(figsize=(10, 10))
    plt.imshow(tile_image, cmap='gray')
    plt.colorbar(label='Elevation')
    plt.title("Single Band View")
    plt.axis('off')

plt.suptitle(f"Tile {example_tile['tile_id']}\nLabel: {tile_label}", fontsize=16)
plt.tight_layout()
plt.show()

# Add this to see individual bands with appropriate labels
if num_bands > 3:
    plt.figure(figsize=(15, 3))

    # Use correct band names for your data
    band_names = ["Red", "Blue", "Green", "Red Edge", "NIR"]

    for i in range(num_bands):
        plt.subplot(1, num_bands, i+1)
        plt.imshow(tile_data[i], cmap='gray')
        plt.title(f"{band_names[i]}")
        plt.axis('off')

    plt.suptitle("Individual Bands")
    plt.tight_layout()
    plt.show()
    # Print some information about the tile
    print(f"Tile ID: {example_tile['tile_id']}")
    print(f"Label: {tile_label}")
    print(f"Coordinates (x,y): ({example_tile['x_coord']}, {example_tile['y_coord']})")
    print(f"Array index: {tile_idx}")
    print(f"Image shape: {tile_image.shape}")
    print(f"Data type: {tile_data.dtype}")
    print(f"Value range: [{tile_data.min()}, {tile_data.max()}]")
else:
    print("No labeled tiles found!")